In [28]:
import pandas as pd
from bs4 import BeautifulSoup
import re

In [29]:
def html_table_to_df(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    # Cari semua tabel
    tables = soup.find_all('table')
    for table in tables:
        rows = table.find_all('tr')
        table_data = []
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 2:
                curah = cols[0].get_text(separator=' ', strip=True)
                kab = cols[1].get_text(separator=' ', strip=True)
                kec = cols[2].get_text(separator=' ', strip=True) if len(cols)>2 else ''
                if ("CURAH HUJAN" in curah and "KOTA" in kab) or (not kab) or ("Cari pada" in kab):
                    continue  # skip header/instruksi/kosong
                table_data.append([curah, kab, kec])
        if table_data:
            df = pd.DataFrame(table_data, columns=['CURAH_HUJAN', 'Kota/Kabupaten', 'KECAMATAN_BAGIAN'])
            # Ekstrak rata2 curah hujan dari range angka:
            def extract_rata2_curah(text):
                match = re.search(r'(\d+)\s*[-–]\s*(\d+)', text)
                if match:
                    return (int(match.group(1)) + int(match.group(2))) / 2
                match = re.search(r'(\d+)\s*mm', text)
                if match:
                    return int(match.group(1))
                return None
            df['CURAH_HUJAN_RATA2'] = df['CURAH_HUJAN'].apply(extract_rata2_curah)
            # Gabung baris dengan kabupaten sama
            result = df.groupby('Kota/Kabupaten', as_index=False)['CURAH_HUJAN_RATA2'].mean()
            return result
    raise ValueError("Tidak ditemukan data tabel relevan dalam file HTML.")

df22_23 = html_table_to_df("(Prakiraan 6 Bulan)Prakiraan  Cuaca Curah Hujan 2022 - 2023.html")
df23_24 = html_table_to_df("(Prakiraan 6 Bulan)Prakiraan  Cuaca Curah Hujan 2023 - 2024.html")

gabung = pd.merge(df22_23, df23_24, on="Kota/Kabupaten", suffixes=('_22_23', '_23_24'))
gabung['CURAH_HUJAN_RATA2_TAHUNAN'] = (gabung['CURAH_HUJAN_RATA2_22_23'] + gabung['CURAH_HUJAN_RATA2_23_24'])/2

gabung.to_csv("curah_hujan_tahunan_jatim.csv", index=False)
print(gabung.head())

  Kota/Kabupaten  CURAH_HUJAN_RATA2_22_23  CURAH_HUJAN_RATA2_23_24  \
0      Bangkalan                 1500.500                    750.5   
1     Banyuwangi                 1937.875                   1000.5   
2         Blitar                 2167.000                   1250.5   
3     Bojonegoro                 1500.500                   1000.5   
4      Bondowoso                 2167.000                   1250.5   

   CURAH_HUJAN_RATA2_TAHUNAN  
0                  1125.5000  
1                  1469.1875  
2                  1708.7500  
3                  1250.5000  
4                  1708.7500  


In [30]:
# raw_text = """
# 1. Kota Surabaya

# Sejarah Tugu Pahlawan adalah sebuah bangunan untuk memperingati peristiwa pertempuran 10 November. Tugu Pahlawan terletak di Surabaya, Jawa Timur. Foto: Bappeko Surabaya
# Posisi pertama ditempati Ibu Kota Provinsi Jawa Timur. Kota Surabaya memiliki jumlah penduduk 2.893.698 jiwa. Kota Pahlawan ini sangat padat penduduk dikarenakan juga memiliki wilayah yang luas.

# 2. Kabupaten Malang
# Urutan kedua terdapat Kabupaten Malang yang memiliki julukan Kota Pendidikan. Kota Malang memiliki total penduduk sebanyak 2.703.175 jiwa.

# 3. Kabupaten Jember
# Posisi ketiga diisi Kabupaten Jember. Kabupaten yang memiliki julukan Kota Seribu Gumuk ini memiliki total jumlah penduduk sebanyak 2.584.771 jiwa.

# 4. Kabupaten Sidoarjo
# Kota Udang dan Bandeng menempati urutan keempat. Jumlah penduduknya total 2.114.588 Jiwa.

# 5. Kabupaten Banyuwangi
# Terletak di ujung timur Pulau Jawa dengan julukan The Sunrise of Java, Kabupaten Banyuwangi menempati urutan kelima. Jumlah penduduk Kabupaten Banyuwangi totalnya 1.744.814 jiwa.

# 6. Kabupaten Kediri
# Kabupaten Kediri menempati urutan keenam dengan jumlah 1.667.450 jiwa. Kediri terkenal sebagai kota penghasil rokok terbesar, selain itu mendapat julukan Kota Tahu.

# 7. Kabupaten Pasuruan
# Posisi ketujuh ditempati Kabupaten Pasuruan dengan total 1.626.029 jiwa. Pasuruan dikenal dengan sebutan Kota Santri karena banyaknya wisata religi di sana. Selain itu, dahulu Pasuruan juga dikenal dengan nama Paravan dan orang Tionghoa menyebut Pasuruan sebagai Yanwang atau Basuluan.

# 8. Kabupaten Lamongan
# Kota Soto menempati urutan kedelapan dengan jumlah penduduk sebanyak 1.386.941 jiwa.

# 9. Kabupaten Jombang
# Kota Santri Jombang berada di posisi kesembilan. Jumlah penduduknya 1.345.886 jiwa.

# 10. Kabupaten Gresik
# Posisi kesepuluh diisi Kabupaten Gresik dengan jumlah penduduk sebanyak 1.344.648 jiwa.

# 11. Kabupaten Bojonegoro
# Urutan kesebelas ada Kota Ledre dengan jumlah penduduk sebanyak 1.322.474 jiwa. Kabupaten Bojonegoro dikenal dengan Kota Ledre karena banyaknya industri makanan pembuat ledre yang dapat dijumpai di sana.

# 12. Kabupaten Blitar
# Kota Proklamator menempati urutan ke-12. Kabupaten tempat Soekarno dimakamkan ini memiliki jumlah penduduk sebanyak 1.249.497 jiwa.

# 13. Kabupaten Tuban
# Posisi ke-13 ada Kota Wali dan Kota Tuak dengan jumlah penduduk sebanyak 1.215.795 jiwa. Julukan Kota Wali diberikan kepada Kabupaten Tuban karena di sini menjadi tempat pusat penyebaran agama Islam. Sedangkan, Kota Tuak diberikan karena banyak penghasil minuman legen atau tuak.

# 14. Kabupaten Probolinggo
# Posisi ke-14 ada Kabupaten Probolinggo. Kota Mangga dan Anggur ini memiliki jumlah penduduk sebanyak 1.163.859 jiwa.

# 15. Kabupaten Mojokerto
# Pada posisi ke-15 ditempati Kota Onde-onde dengan penduduk sebanyak 1.141.516 jiwa.

# 16. Kabupaten Lumajang
# Pada posisi ke-16 ada Kota Pisang dengan penduduk sebanyak 1.147.261 jiwa. Alasan dijuluki Kota Pisang karena Lumajang terkenal dengan banyaknya jenis pisang. Salah satunya pisang agung yang memiliki ukuran sangat besar.

# 17. Kabupaten Nganjuk
# Selanjutnya, posisi ke-17 terdapat Kota Angin dengan penduduk sebanyak 1.124.247 jiwa. Kabupaten Nganjuk dijuluki Kota Angin karena angin kencang dapat dirasakan setiap hari.

# 18. Kabupaten Tulungagung
# Urutan ke-18 terdapat Kabupaten Tulungagung yang terkenal sebagai daerah penghasil marmer terbesar di Indonesia atau biasa disebut Sweden van Java. Jumlah penduduk di Kabupaten Tulungagung sebanyak 1.113.973 jiwa.

# 19. Kabupaten Sumenep
# Posisi ke-19 terdapat Kabupaten Sumenep dengan jumlah penduduk sebanyak 1.143.295 jiwa. Selain terkenal pantainya, Kabupaten Sumenep juga terkenal dengan sentra pembuatan keris.

# 20. Kabupaten Bangkalan
# Disusul Kabupaten Bangkalan di urutan ke-20. Daerah paling barat di Pulau Jawa ini memiliki jumlah penduduk sebanyak 1.101.556 jiwa.

# 21. Kabupaten Sampang
# Satu lagi daerah di Pulau Madura masuk posisi ke-21, yaitu Kabupaten Sampang. Jumlah penduduknya sebanyak 992.210 jiwa.

# 22. Kabupaten Ponorogo
# Selanjutnya, urutan ke-21 ada Kabupaten Ponorogo dengan julukan Kota Reog. Jumlah penduduknya sebanyak 972.582 jiwa.

# 23. Kabupaten Ngawi
# Kota Bambu yaitu Kabupaten Ngawi menyusul di urutan ke-23 dengan jumlah penduduk sebanyak 881.393 jiwa.

# 24. Kabupaten Pamekasan
# Kota Batik yaitu Kabupaten Pamekasan berada di urutan ke-24 dengan jumlah penduduk yaitu 862.009 jiwa.

# 25. Kota Malang
# Kota Apel berada di urutan ke-25 dengan jumlah penduduk sebanyak 847.182 jiwa.

# 26. Kabupaten Bondowoso
# Lanjut urutan 26 ada Kota Tape atau Kabupaten Bondowoso dengan jumlah penduduk sebanyak 784.192 jiwa.

# 27. Kabupaten Madiun
# Kota Brem dan Kota Pecel yaitu Kabupaten Madiun berada di posisi ke-27 dengan jumlah penduduk yaitu 765.135 jiwa.

# 28. Kabupaten Trenggalek
# Terdapat Kabupaten Trenggalek atau dikenal Kota Gaplek di urutan ke-28. Jumlah penduduknya sebanyak 744.358 jiwa.

# 29. Kabupaten Situbondo
# Surga Burung menjadi julukan Kabupaten Situbondo. Kabupaten ini menempati posisi ke-29 dengan jumlah penduduk sebanyak 694.081 jiwa.

# 30. Kabupaten Magetan
# The Nice of Java merupakan julukan Kabupaten Magetan karena wisata alam terkenal Telaga Sarangan. Jumlah penduduk di Kabupaten Magetan adalah 682.466 jiwa.

# 31. Kabupaten Pacitan
# Kabupaten Pacitan terkenal wisata alamnya yang indah, mulai dari pantai sampai gua. Karena itu Pacitan mendapat julukan Kota 1001 Gua. Jumlah penduduk di Kabupaten Pacitan sebanyak 596.649 jiwa.

# 32. Kota Kediri
# Kota Kediri menempati urutan ke-32 dengan jumlah 290.836 jiwa. Kediri terkenal sebagai kota penghasil rokok terbesar, selain itu mendapat julukan Kota Tahu.

# 33. Kota Probolinggo
# Posisi ke-33 terdapat Kota Probolinggo dengan julukan Kota Mangga dan Anggur. Jumlah penduduknya sebanyak 245.174 jiwa.

# 34. Kota Batu
# Kota Batu menempati urutan ke-32 dengan jumlah 218.802 jiwa. Batu terkenal sebagai Kota Apel dan memiliki tempat wisata terkenal yang selalu ramai pengunjung.

# 35. Kota Pasuruan
# Posisi 35 ditempati Kota Pasuruan dengan total penduduk 213.450 jiwa. Pasuruan dikenal dengan sebutan Kota Santri karena banyak wisata religi di sana. Selain itu, dahulu Pasuruan juga dikenal dengan nama Paravan dan orang Tionghoa menyebut Pasuruan sebagai Yanwang atau Basuluan.

# 36. Kota Madiun
# Kota Brem dan Kota Pecel yaitu Kabupaten Madiun berada di posisi 36 dengan jumlah penduduk 201.460 jiwa.

# 37. Kota Blitar
# Urutan ke-37 ada Kota Blitar dengan jumlah penduduk sebanyak 153.541 jiwa.

# 38. Kota Mojokerto
# Posisi terakhir ditempati Kota Onde-onde dengan penduduk sebanyak 135.414 jiwa.
# """

# pattern = r"(Kota|Kabupaten)\s([A-Za-z ]+).*?(\d[\d\.]+)\sjiwa"
# matches = re.findall(pattern, raw_text, flags=re.IGNORECASE)

# data = []
# seen = set()
# for tipe, nama, jumlah in matches:
#     kabkot = f"{tipe} {nama.strip()}"
#     jumlah_penduduk = int(jumlah.replace('.', ''))
#     if kabkot not in seen:
#         data.append({'Kota/Kabupaten': kabkot, 'Penduduk': jumlah_penduduk})
#         seen.add(kabkot)

# df = pd.DataFrame(data)
# print(df)
# df.to_csv('kepadatan_penduduk_jatim_raw.csv', index=False)


In [31]:
# julukan_map = {
#     "Kota Provinsi Jawa Timur": "Kota Surabaya",
#     "Kabupaten Malang yang memiliki julukan Kota Pendidikan": "Kabupaten Malang",
#     "Kabupaten Jember": "Kabupaten Jember",
#     "Kota Udang dan Bandeng menempati urutan keempat": "Kabupaten Sidoarjo",
#     "Kabupaten Banyuwangi menempati urutan kelima": "Kabupaten Banyuwangi",
#     "Kabupaten Kediri menempati urutan keenam dengan jumlah": "Kabupaten Kediri",
#     "Kabupaten Pasuruan dengan total": "Kabupaten Pasuruan",
#     "Kota Soto menempati urutan kedelapan dengan jumlah penduduk sebanyak": "Kabupaten Lamongan",
#     "Kota Santri Jombang berada di posisi kesembilan": "Kabupaten Jombang",
#     "Kabupaten Gresik dengan jumlah penduduk sebanyak": "Kabupaten Gresik",
#     "Kota Ledre dengan jumlah penduduk sebanyak": "Kabupaten Bojonegoro",
#     "Kota Proklamator menempati urutan ke": "Kabupaten Blitar",
#     "Kota Wali dan Kota Tuak dengan jumlah penduduk sebanyak": "Kabupaten Tuban",
#     "Kabupaten Probolinggo": "Kabupaten Probolinggo",
#     "Kota Onde": "Kabupaten Mojokerto",
#     "Kota Pisang dengan penduduk sebanyak": "Kabupaten Lumajang",
#     "Kota Angin dengan penduduk sebanyak": "Kabupaten Nganjuk",
#     "Kabupaten Tulungagung yang terkenal sebagai daerah penghasil marmer terbesar di Indonesia atau biasa disebut Sweden van Java": "Kabupaten Tulungagung",
#     "Kabupaten Sumenep dengan jumlah penduduk sebanyak": "Kabupaten Sumenep",
#     "Kabupaten Bangkalan di urutan ke": "Kabupaten Bangkalan",
#     "Kabupaten Sampang": "Kabupaten Sampang",
#     "Kabupaten Ponorogo dengan julukan Kota Reog": "Kabupaten Ponorogo",
#     "Kota Bambu yaitu Kabupaten Ngawi menyusul di urutan ke": "Kabupaten Ngawi",
#     "Kota Batik yaitu Kabupaten Pamekasan berada di urutan ke": "Kabupaten Pamekasan",
#     "Kota Apel berada di urutan ke": "Kota Malang",
#     "Kota Tape atau Kabupaten Bondowoso dengan jumlah penduduk sebanyak": "Kabupaten Bondowoso",
#     "Kota Brem dan Kota Pecel yaitu Kabupaten Madiun berada di posisi ke": "Kabupaten Madiun",
#     "Kabupaten Trenggalek atau dikenal Kota Gaplek di urutan ke": "Kabupaten Trenggalek",
#     "Kabupaten Situbondo": "Kabupaten Situbondo",
#     "Kabupaten Magetan karena wisata alam terkenal Telaga Sarangan": "Kabupaten Magetan",
#     "Kabupaten Pacitan terkenal wisata alamnya yang indah": "Kabupaten Pacitan",
#     "Kota Kediri menempati urutan ke": "Kota Kediri",
#     "Kota Probolinggo dengan julukan Kota Mangga dan Anggur": "Kota Probolinggo",
#     "Kota Batu menempati urutan ke": "Kota Batu",
#     "Kota Pasuruan dengan total penduduk": "Kota Pasuruan",
#     "Kota Brem dan Kota Pecel yaitu Kabupaten Madiun berada di posisi": "Kota Madiun",
#     "Kota Blitar dengan jumlah penduduk sebanyak": "Kota Blitar",
#     "Kota Mojokerto": "Kota Mojokerto",
# }


In [32]:
# df = pd.read_csv("kepadatan_penduduk_jatim_raw.csv")

# df["Kota/Kabupaten_BPS"] = df["Kota/Kabupaten"].map(julukan_map)
# # opsi jika ada panggilan yg belum di-mapping → tetap pakai nama aslinya
# df["Kota/Kabupaten_BPS"] = df["Kota/Kabupaten_BPS"].fillna(df["Kota/Kabupaten"])

# print(df[["Kota/Kabupaten", "Kota/Kabupaten_BPS", "Penduduk"]])
# df[["Kota/Kabupaten_BPS", "Penduduk"]].to_csv("data_kepadatan_bersih.csv", index=False)


In [ ]:
df = pd.read_csv("Kasus_Penyaki_Menurut_Kabupaten_Kota_dan_Jenis_Penyakit_di_Provinsi_Jawa_Timur_2023.csv")

# Cari kolom yang mengandung 'DBD'
dbd_col = [col for col in df.columns if 'DBD' in col][0]

# Buat dataframe baru dengan 2 kolom: nama & DBD
df_dbd = df[["Kabupaten/Kota", dbd_col]].copy()
# Rename kolom agar jelas
df_dbd = df_dbd.rename(columns={"Kabupaten/Kota": "Kota/Kabupaten", dbd_col: "Kasus_DBD_per_100rb"})

print(df_dbd.head())

# Simpan ke CSV rekap DBD
df_dbd.to_csv("dbd_jatim_2023.csv", index=False)

KeyError: "['Kota/Kabupaten'] not in index"

In [34]:
df = pd.read_csv("Jumlah_dan_Persentase_Penduduk_Miskin_Menurut_Kabupaten_Kota_di_Provinsi_Jawa_Timur_2023.csv")

# Pilih kolom penting
col_kab = "Kabupaten/Kota"
col_persen = "Persentase Penduduk Miskin - Maret"

df_miskin = df[[col_kab, col_persen]].copy()
df_miskin.columns = ["Kota/Kabupaten", "Persen_Miskin"]

# Buang NaN pada kolom nama terlebih dulu
df_miskin = df_miskin[df_miskin['Kota/Kabupaten'].notna()]

# Baru filter baris yang hanya memuat nama kabupaten/kota (bukan Jawa Timur, Catatan, dsb)
mask = df_miskin['Kota/Kabupaten'].str.contains('Kota|Kabupaten')
df_miskin = df_miskin[mask]

# Jika ada NaN di jumlah/persen, buang juga
df_miskin = df_miskin.dropna(subset=["Persen_Miskin"])

# Convert ke float
df_miskin['Persen_Miskin'] = df_miskin['Persen_Miskin'].astype(float)

print(df_miskin.head())
df_miskin.to_csv("data_penduduk_miskin_bersih.csv", index=False)

          Kota/Kabupaten  Persen_Miskin
0      Kabupaten Pacitan          13.65
1     Kabupaten Ponorogo           9.53
2   Kabupaten Trenggalek          10.63
3  Kabupaten Tulungagung           6.53
4       Kabupaten Blitar           8.69


In [35]:
df_kepadatan_raw = pd.read_csv("Penduduk_Laju_Pertumbuhan_Penduduk_Distribusi_Persentase_Penduduk_Kepadatan_Penduduk,_Rasio_Jenis_Kelamin_Penduduk_Menurut_Kabupaten_Kota_di_Provinsi_Jawa_Timur_2023.csv")

# Pilih kolom: nama kabupaten/kota dan kepadatan penduduk per km2
df_kepadatan = df_kepadatan_raw[["Kabupaten/Kota", "Kepadatan Penduduk per km persegi (Km2)"]].copy()
df_kepadatan.columns = ["Kota/Kabupaten", "Kepadatan_Penduduk_per_km2"]

# Buang baris dengan nilai kosong
df_kepadatan = df_kepadatan.dropna()

# Hapus baris "Jawa Timur"
df_kepadatan = df_kepadatan[df_kepadatan['Kota/Kabupaten'] != 'Jawa Timur']

# Tambahkan prefix "Kabupaten" untuk baris yang tidak memiliki "Kota"
df_kepadatan['Kota/Kabupaten'] = df_kepadatan['Kota/Kabupaten'].apply(
    lambda x: x if 'Kota' in x else f'Kabupaten {x}'
)

# Convert kepadatan ke numeric (jika ada karakter tambahan)
df_kepadatan['Kepadatan_Penduduk_per_km2'] = pd.to_numeric(df_kepadatan['Kepadatan_Penduduk_per_km2'], errors='coerce')
df_kepadatan = df_kepadatan.dropna(subset=['Kepadatan_Penduduk_per_km2'])

print("Data kepadatan penduduk per km2 yang sudah dibersihkan:")
print(df_kepadatan.head())
print(f"\nJumlah data: {len(df_kepadatan)} kabupaten/kota")

# Simpan hasil cleaning
df_kepadatan.to_csv("data_kepadatan_bersih.csv", index=False)
print("\nFile tersimpan: data_kepadatan_bersih.csv")

Data kepadatan penduduk per km2 yang sudah dibersihkan:
          Kota/Kabupaten  Kepadatan_Penduduk_per_km2
0      Kabupaten Pacitan                       410.0
1     Kabupaten Ponorogo                       676.0
2   Kabupaten Trenggalek                       593.0
3  Kabupaten Tulungagung                       968.0
4       Kabupaten Blitar                       718.0

Jumlah data: 38 kabupaten/kota

File tersimpan: data_kepadatan_bersih.csv


In [36]:
print(df_dbd.columns)
print(df_hujan.columns)
print(df_miskin.columns)
print(df_kepadatan.columns)


Index(['Kabupaten/Kota', 'Kasus_DBD_per_100rb'], dtype='object')
Index(['Kota/Kabupaten', 'CURAH_HUJAN_RATA2_22_23', 'CURAH_HUJAN_RATA2_23_24',
       'CURAH_HUJAN_RATA2_TAHUNAN'],
      dtype='object')
Index(['Kota/Kabupaten', 'Persen_Miskin'], dtype='object')
Index(['Kota/Kabupaten', 'Kepadatan_Penduduk_per_km2'], dtype='object')


In [38]:
df_dbd = pd.read_csv("dbd_jatim_2023.csv")                   # Data kasus DBD
df_hujan = pd.read_csv("curah_hujan_tahunan_jatim.csv") # Data curah hujan tahunan
df_miskin = pd.read_csv("data_penduduk_miskin_bersih.csv")   # Data penduduk miskin
df_kepadatan = pd.read_csv("data_kepadatan_bersih.csv") # Data kepadatan penduduk
# df_kepadatan = pd.read_csv('Penduduk_Laju_Pertumbuhan_Penduduk_Distribusi_Persentase_Penduduk_Kepadatan_Penduduk,_Rasio_Jenis_Kelamin_Penduduk_Menurut_Kabupaten_Kota_di_Provinsi_Jawa_Timur_2023.csv')
# # Merge berurutan (pakai inner agar hanya kabupaten/kota yg ada di semua data)
data_final = df_dbd.merge(df_hujan, on="Kota/Kabupaten")\
                   .merge(df_miskin, on="Kota/Kabupaten")\
                   .merge(df_kepadatan, on="Kota/Kabupaten")

data_final.to_csv("data_spasial_jatim_final.csv", index=False)
print(data_final.head())


KeyError: 'Kota/Kabupaten'